In [1]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [2]:
# Install latest Chrome, ChromeDriver, and Selenium
!apt-get update -qq
!apt-get install -yqq unzip
!apt-get install -yqq wget curl

# Install latest Google Chrome
!wget -q -O /tmp/google-chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -yqq /tmp/google-chrome.deb

# Get Chrome version
!google-chrome --version

# Match ChromeDriver version
!CHROME_VERSION=$(google-chrome --version | grep -oP '\d+' | head -1) \
 && DRIVER_VERSION=$(curl -s "https://googlechromelabs.github.io/chrome-for-testing/LATEST_RELEASE") \
 && wget -q -O /tmp/chromedriver.zip "https://storage.googleapis.com/chrome-for-testing-public/$DRIVER_VERSION/linux64/chromedriver-linux64.zip" \
 && unzip /tmp/chromedriver.zip -d /usr/local/bin/ \
 && mv /usr/local/bin/chromedriver-linux64/chromedriver /usr/local/bin/chromedriver \
 && chmod +x /usr/local/bin/chromedriver

# Verify installations
!chromedriver --version
!pip install -q selenium


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Some index files failed to download. They have been ignored, or old ones used instead.
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data l

In [10]:
import copy
import random
import re
import time
import requests
import os
from typing import Dict, Any, Optional, List # for formating data
from bs4 import BeautifulSoup #scraping
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.remote.webdriver import WebDriver
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from datetime import datetime

In [56]:
import os
import re
import time
import random
import copy
import requests
from datetime import datetime
from typing import Optional, Dict, Any

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ==========================================
# UTILITY FUNCTIONS
# ==========================================

def init_driver(headless: bool = True) -> webdriver.Chrome:
    """Initializes and configures the Selenium Chrome WebDriver."""
    chrome_options = Options()

    if headless:
        chrome_options.add_argument('--headless')

    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument("--blink-settings=imagesEnabled=false")
    chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36')
    chrome_options.add_argument("webdriver.chrome.driver=/usr/lib/chromium-browser/chromedriver")

    driver = webdriver.Chrome(options=chrome_options)
    return driver

def clean_and_convert_to_int(text: Optional[str]) -> Optional[int]:
    """Removes all non-digit characters from a string and converts it to an integer."""
    if not isinstance(text, str) or not text:
        return None
    try:
        cleaned_string = re.sub(r'\D', '', text)
        return int(cleaned_string)
    except (ValueError, TypeError):
        print(f"Warning: Could not convert '{text}' to an integer.")
        return None

def safe_get_text(element, default=None, strip=True):
    """Safely extracts text from a BeautifulSoup element."""
    try:
        return element.get_text(strip=strip) if element else default
    except Exception:
        return default

def safe_get_attr(element, attr, default=None):
    """Safely gets an attribute from a BeautifulSoup element."""
    try:
        return element.get(attr) if element and hasattr(element, 'get') else default
    except Exception:
        return default

def _download_images(image_urls: list, base_folder: str, sub_folder: str):
    """Helper function to handle downloading images safely."""
    if not image_urls:
        return

    folder_path = os.path.join(base_folder, sub_folder)
    try:
        os.makedirs(folder_path, exist_ok=True)
        for index, url in enumerate(image_urls, start=1):
            try:
                response = requests.get(url, timeout=10)
                if response.status_code == 200:
                    file_path = os.path.join(folder_path, f"{index}.jpg")
                    with open(file_path, "wb") as file:
                        file.write(response.content)
            except Exception as e:
                print(f"Error downloading individual image {url}: {e}")
    except Exception as e:
        print(f"Error creating directory {folder_path}: {e}")

# ==========================================
# SCRAPING LOGIC
# ==========================================

def get_soup(url: str, headless: bool = True, max_retries: int = 20, target_selector: str = None) -> Optional[BeautifulSoup]:
    """Loads the page, handles Cloudflare blocks, and returns the parsed HTML."""
    for attempt in range(1, max_retries + 1):
        driver = None
        try:
            driver = init_driver(headless=headless)
            print(f"Attempt {attempt}/{max_retries}: Navigating to {url}...")
            driver.get(url)

            # 1. Immediate Cloudflare Check
            if "Just a moment" in driver.title or "Cloudflare" in driver.title:
                print(f" [!] Blocked by Cloudflare on attempt {attempt}. Exiting to retry...")
                raise Exception("Cloudflare Blocked")

            # 2. Wait for main data to render
            if target_selector:
                print(" -> Waiting for data to render...")
                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, target_selector))
                )

            # Success: Get soup and return
            print(" [OK] Data retrieved successfully.")
            time.sleep(random.uniform(0.5, 1.0))
            soup = BeautifulSoup(driver.page_source, "html.parser")

            driver.quit()
            return soup

        except Exception as e:
            print(f" [X] Error: {e}")
            if driver:
                driver.quit()

            # Sleep before retry if attempts remain
            if attempt < max_retries:
                wait_time = random.uniform(1.0, 3.0)
                print(f" -> Resting for {wait_time:.2f}s before re-initializing driver...\n")
                time.sleep(wait_time)
            else:
                print(" [!!!] Max retries reached. Could not bypass Cloudflare.")

    return None

def _parse_listing_data(soup: BeautifulSoup) -> Dict[str, Any]:
    """Parses the HTML soup to extract car listing data safely."""
    data = {'datetime': datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

    # ==========================================
    # 1. POST DATA (Listing Specifics)
    # ==========================================
    post = {}

    # --- Title & New/Used ---
    try:
        title_text = safe_get_text(soup.select_one("#vehicle-title"))
        if title_text:
            title_parts = title_text.split(' ', 1)
            post["new_used"] = title_parts[0] if title_parts else None
            post["title"] = title_parts[1] if len(title_parts) > 1 else None
        else:
            post["new_used"], post["title"] = None, None
    except Exception:
        post["new_used"], post["title"] = None, None

    # --- Mileage, Price, Monthly Payment ---
    post["mileage"] = clean_and_convert_to_int(safe_get_text(soup.select_one("div.msrp")))
    post["price"] = clean_and_convert_to_int(safe_get_text(soup.select_one(".list-price")))
    post["monthly_payment"] = clean_and_convert_to_int(safe_get_text(soup.select_one('fuse-stack[data-qa="monthly-payment"] fuse-button')))

    # --- Basics & VIN ---
    basic_dict = {}
    try:
        subtitle_text = safe_get_text(soup.select_one('section#features-and-specs div.subtitle'), default="")
        vin_match = re.search(r'VIN:\s*([A-Z0-9]+)', subtitle_text)
        stock_match = re.search(r'Stock #:\s*([A-Z0-9]+)', subtitle_text)
        if vin_match: basic_dict["VIN"] = vin_match.group(1)
        if stock_match: basic_dict["Stock Number"] = stock_match.group(1)

        for entry in soup.select('div.basics li[data-qa="basics-entry"]'):
            text = safe_get_text(entry)
            if text:
                match = re.search(r'(?i)(.*)\s+(exterior color|interior color|fuel type|engine|mpg|drivetrain|transmission)$', text)
                if match:
                    val, key_raw = match.group(1).strip(), match.group(2).strip().title()
                    key = "MPG" if key_raw.lower() == "mpg" else key_raw
                    basic_dict[key] = val
                else:
                    basic_dict["Feature"] = text
    except Exception:
        pass

    post["basic_desc"] = basic_dict if basic_dict else None

    # --- Feature Terms ---
    try:
        feature_dict = {}
        for block in soup.select('div.highlight-feature'):
            key = safe_get_text(block.select_one('h3.features-spec-heading'))
            if key:
                values = [safe_get_text(li) for li in block.select('li[data-qa="spec-value"]')]
                feature_dict[key] = values
        post["feature_des"] = feature_dict if feature_dict else None
    except Exception:
        post["feature_des"] = None

    # --- User History ---
    try:
        user_history_dict = {}
        history_section = soup.select_one('section#vehicle_history_report')
        if history_section:
            for item in history_section.select('fuse-list ul li span'):
                text_lower = safe_get_text(item, default="").lower()
                if 'accident' in text_lower or 'damage' in text_lower:
                    user_history_dict["Accidents or damage"] = "None reported" if text_lower.startswith('no ') else "At least 1 accident or damage reported"
                elif 'owner' in text_lower:
                    user_history_dict["1-owner vehicle"] = "Yes" if '1 owner' in text_lower else "No"
                elif 'personal use' in text_lower:
                    user_history_dict["Personal use only"] = "Yes"
                elif 'recall' in text_lower:
                    user_history_dict["Open recall"] = "None reported" if 'no ' in text_lower else "At least 1 open recall reported"
                elif 'title' in text_lower:
                    user_history_dict["Clean Title"] = "Yes" if 'clean' in text_lower else "No"
        post["user_history_des"] = user_history_dict if user_history_dict else None
    except Exception:
        post["user_history_des"] = None

    # --- Warranty ---
    try:
        warranty_dict = {}
        warranty_terms = soup.select_one('section.sds-page-section.warranty_section dl.fancy-description-list')
        if warranty_terms:
            for term in warranty_terms.find_all('dt', recursive=False):
                key = safe_get_text(term)
                value_tag = term.find_next_sibling('dd')
                val = safe_get_text(value_tag)
                if key and val:
                    warranty_dict[key] = None if val in ['–', '-', '—'] else val

        warranty_dl = soup.select_one('fuse-popover#cpo-popover dl')
        if warranty_dl:
            for dt_tag in warranty_dl.find_all('dt'):
                key = safe_get_text(dt_tag)
                dd_tag = dt_tag.find_next_sibling('dd')
                val = ' '.join(safe_get_text(dd_tag, default="").split())
                if key and val:
                    warranty_dict[key] = None if val in ['-', '–', '—'] else val

        post["warranty_des"] = warranty_dict if warranty_dict else None
    except Exception:
        post["warranty_des"] = None

    # --- Car Images ---
    try:
        imgs = [safe_get_attr(img, 'src').split('?')[0] for img in soup.select('fuse-gallery-grid#gallery img')[:5] if safe_get_attr(img, 'src')]
        post['image'] = imgs if imgs else None
        if post['image']:
            vin = basic_dict.get('VIN') or basic_dict.get('Stock Number') or 'Unknown_VIN'
            _download_images(post['image'], "downloaded_images/post_images", vin)
    except Exception:
        post['image'] = None

    data['post'] = post

    # ==========================================
    # 2. SELLER DATA
    # ==========================================
    seller = {}

    seller['seller_name'] = safe_get_text(soup.select_one('div.dealer-info h3'))

    # Seller Link
    try:
        raw_link = safe_get_attr(soup.select_one('div.dealer-info fuse-stack.rating-stack a'), 'href')
        if raw_link:
            seller['seller_link'] = 'https://www.cars.com' + raw_link if raw_link.startswith('/') else raw_link
            url_parts = [p for p in seller['seller_link'].split('#')[0].split('/') if p]
            seller['seller_key'] = url_parts[-1] if url_parts else None
        else:
            seller['seller_link'], seller['seller_key'] = None, None
    except Exception:
        seller['seller_link'], seller['seller_key'] = None, None

    seller['destination'] = safe_get_text(soup.select_one('div.dealer-info div.map-link a'))
    seller['seller_website'] = safe_get_attr(soup.select_one('div.dealer-info div.website a'), 'href')

    # Seller Rating
    try:
        rating_tag = soup.select_one('fuse-stack.rating-stack fuse-rating')
        seller['seller_rating'] = float(safe_get_attr(rating_tag, 'rating')) if safe_get_attr(rating_tag, 'rating') else None

        review_text = safe_get_text(rating_tag.find('a') if rating_tag else None)
        if review_text:
            match = re.search(r'[\d,]+', review_text)
            seller['seller_rating_count'] = int(match.group(0).replace(',', '')) if match else None
        else:
            seller['seller_rating_count'] = None
    except Exception:
        seller['seller_rating'], seller['seller_rating_count'] = None, None

    # Seller Highlights & Description
    try:
        highlights = [safe_get_text(li) for li in soup.select('div.highlights-tab fuse-list ul li') if safe_get_text(li)]
        seller['highlights'] = highlights if highlights else None
    except Exception:
        seller['highlights'] = None

    seller['description'] = safe_get_text(soup.select_one('div.about-tab__carsons-summary-body cars-line-clamp p'))

    # Seller Images
    try:
        seller_imgs = [safe_get_attr(img, 'src').split('?')[0] for img in soup.select('card-gallery.dealership-gallery img')[:2] if safe_get_attr(img, 'src')]
        seller['image'] = seller_imgs if seller_imgs else None
        if seller['image']:
            seller_folder = seller.get('seller_key') or 'Unknown_Seller'
            _download_images(seller['image'], "downloaded_images/seller_images", seller_folder)
    except Exception:
        seller['image'] = None

    data['seller'] = seller

    # ==========================================
    # 3. CAR MODEL DATA
    # ==========================================
    car = {}

    # Car Rating & Count
    try:
        rating_text = safe_get_text(soup.select_one('section#consumer-reviews div.rating-out-of span.rating-count'))
        car['car_rating'] = float(rating_text) if rating_text else None

        rating_out_of_div = soup.select_one('.rating-out-of')
        count_text = safe_get_text(rating_out_of_div.find_next_sibling('div') if rating_out_of_div else None)
        count_match = re.search(r'Based on ([\d,]+) reviews', count_text) if count_text else None
        car['car_rating_count'] = int(count_match.group(1).replace(',', '')) if count_match else None
    except Exception:
        car['car_rating'], car['car_rating_count'] = None, None

    # Ratings Breakdown
    try:
        ratings = {}
        breakdown_container = soup.select_one('section#consumer-reviews div.ratings-breakdown')
        if breakdown_container:
            for review_div in breakdown_container.select('div.review-text'):
                value_str = safe_get_text(review_div.find('strong'))
                if value_str:
                    category_name = safe_get_text(review_div).replace(value_str, '').strip()
                    try:
                        ratings[category_name] = float(value_str)
                    except ValueError:
                        ratings[category_name] = value_str
        car['ratings'] = ratings if ratings else None
    except Exception:
        car['ratings'] = None

    # Percentage Recommended
    try:
        subtitle_text = safe_get_text(soup.select_one('section#consumer-reviews div.subtitle'))
        match = re.search(r'(\d+(?:\.\d+)?)%', subtitle_text) if subtitle_text else None
        car['percentage_recommend'] = float(match.group(1)) if match else None
    except Exception:
        car['percentage_recommend'] = None

    # Links & Model (Review Button Parsing)
    try:
        raw_url = safe_get_attr(soup.select_one('fuse-button.review-btn-secondary[href*="write-a-review"]'), 'href')
        if raw_url:
            raw_url = 'https://www.cars.com' + raw_url if raw_url.startswith('/') else raw_url
            car['car_link'] = raw_url.replace('write-a-review/', '')

            url_parts = [p for p in raw_url.split('/') if p]
            if 'write-a-review' in url_parts:
                car['car_model'] = url_parts[url_parts.index('write-a-review') - 1]
                car['brand'] = car['car_model'].split('-')[0].replace('_', ' ').title()

                parts = car['car_model'].split('-')
                if parts[-1].isdigit() and len(parts[-1]) == 4:
                    year = parts.pop()
                    car['car_name'] = f"{year} {' '.join(parts).title()}"
                else:
                    car['car_name'] = " ".join(parts).title()
            else:
                car['car_model'], car['brand'], car['car_name'] = None, None, None

            car['review_link'] = car['car_link'] + 'consumer-reviews/?page_size=40' if car['car_link'] else None
        else:
            car['car_model'], car['car_link'], car['review_link'], car['brand'], car['car_name'] = None, None, None, None, None
    except Exception:
        car['car_model'], car['car_link'], car['review_link'], car['brand'], car['car_name'] = None, None, None, None, None

    # ------ REVIEWS --------
    listing_reviews_terms = soup.select('div.consumer-review-container')
    reviews = []

    if listing_reviews_terms:
        for review_term in listing_reviews_terms:
            review_data = {}

            # 1. Overall Rating
            overall_rating_tag = review_term.select_one('fuse-rating, spark-rating')
            rating_val = safe_get_attr(overall_rating_tag, 'rating')
            review_data['overall_rating'] = float(rating_val) if rating_val else None

            # 2. Time (Date)
            time_tag = review_term.select_one('.review-date') or review_term.select_one('.review-byline > div:nth-child(1)')
            review_data['time'] = safe_get_text(time_tag)

            # 3. User Name & Location
            author_tag = review_term.select_one('.author-details > div:first-child') or review_term.select_one('.review-byline > div:nth-child(2)')
            byline_text = safe_get_text(author_tag)

            review_data['user_name'], review_data['from'] = None, None
            if byline_text:
                if byline_text.startswith("By "):
                    byline_text = byline_text[3:]
                if " from " in byline_text:
                    parts = byline_text.split(" from ", 1)
                    review_data['user_name'], review_data['from'] = parts[0].strip(), parts[1].strip()
                elif " on " in byline_text:
                    parts = byline_text.split(" on ", 1)
                    review_data['user_name'], review_data['from'] = parts[0].strip(), parts[1].strip()
                else:
                    review_data['user_name'] = byline_text.strip()

            # 4. Title
            review_data['title'] = safe_get_text(review_term.select_one('h3.title'))

            # 5. Rating Breakdown
            ratings_breakdown = {}
            breakdown_stacks = review_term.select('.ratings-breakdown fuse-stack')

            if breakdown_stacks:
                for stack in breakdown_stacks:
                    text_div = stack.select_one('.review-text')
                    val_str = safe_get_text(text_div.find('strong') if text_div else None)
                    if val_str:
                        key = safe_get_text(text_div).replace(val_str, '').strip()
                        try:
                            ratings_breakdown[key] = float(val_str)
                        except ValueError:
                            ratings_breakdown[key] = val_str
            else:
                breakdown_list = review_term.select_one('.review-breakdown--list')
                if breakdown_list:
                    for item in breakdown_list.select('li'):
                        key = safe_get_text(item.select_one('.sds-definition-list__display-name'))
                        value_str = safe_get_text(item.select_one('.sds-definition-list__value'))
                        if key and value_str:
                            try:
                                ratings_breakdown[key] = float(value_str)
                            except ValueError:
                                ratings_breakdown[key] = value_str

            review_data['ratings_breakdown'] = ratings_breakdown if ratings_breakdown else None

            # 6. Review Body
            clamp_tag = review_term.select_one('cars-line-clamp')
            if clamp_tag:
                feedback_div = clamp_tag.select_one('.review-feedback')
                if feedback_div:
                    feedback_div.extract()
                review_data['review'] = safe_get_text(clamp_tag)
            else:
                review_data['review'] = safe_get_text(review_term.select_one('p.review-body'))

            # Skip empty entries
            if not review_data['review'] and not review_data['title']:
                continue

            reviews.append(review_data)
    else:
        reviews = None

    car['reviews'] = reviews
    data['car'] = car

    return data

def _parse_seller_data(soup: BeautifulSoup, data: Dict[str, Any]) -> Dict[str, Any]:
    """Parses extended seller data from the specific dealership page."""
    output_data = copy.deepcopy(data)

    # Phone Info
    phone_data = {}
    for phone in soup.select('div.dealer-phone'):
        phone_type = safe_get_text(phone.select_one('span.phone-number-title'), default='Unknown')
        phone_number = safe_get_text(phone.select_one('a.phone-number'))
        if phone_type and phone_number:
            phone_data[phone_type] = phone_number
    output_data['seller']['phone_info'] = phone_data

    # Hours Info
    hours_data = {}
    for row in soup.select('table.dealer-hours tr'):
        cells = row.find_all('td')
        if len(cells) == 2:
            key = safe_get_text(cells[0]).replace(':', '')
            value = safe_get_text(cells[1])
            hours_data[key] = value
    output_data['seller']['hours'] = hours_data

    return output_data

def scrape_post_website(url: str, headless: bool = True) -> Optional[Dict[str, Any]]:
    """Scrapes main car listing."""
    try:
        soup = get_soup(url, headless=headless, target_selector="div.reviews-collection div.consumer-review-container div.ratings-breakdown")
        if soup:
            return _parse_listing_data(soup)
    except Exception as e:
        print(f"An error occurred while scraping {url}: {e}")
    return None

def scrape_seller_website(url: str, data: Dict[str, Any], headless: bool = True) -> Optional[Dict[str, Any]]:
    """Scrapes secondary dealership page."""
    try:
        soup = get_soup(url, headless=headless, target_selector="div.dealer-contact-section")
        if soup:
            return _parse_seller_data(soup, data)
    except Exception as e:
        print(f"An error occurred while scraping seller page {url}: {e}")
    return data # Safe fallback to return the already parsed data if this fails

def scrape_full_info(url: str, headless: bool = True) -> Optional[Dict[str, Any]]:
    """Orchestrates full scraping sequence."""
    scraped_data = scrape_post_website(url, headless=headless)

    if scraped_data and scraped_data.get('seller', {}).get('seller_link'):
        seller_url = scraped_data['seller']['seller_link']
        scraped_data = scrape_seller_website(seller_url, scraped_data, headless=headless)

    return scraped_data

In [51]:
A = "https://www.cars.com/vehicledetail/0a950831-91f8-463a-af7b-782b6ceacc19/?attribution_type=isa"

scrape_full_info(A)

Attempt 1/5: Navigating to https://www.cars.com/vehicledetail/0a950831-91f8-463a-af7b-782b6ceacc19/?attribution_type=isa...
 -> Waiting for data to render...
 [OK] Data retrieved successfully.
Attempt 1/5: Navigating to https://www.cars.com/dealers/209950/berman-subaru-of-chicago/#Reviews...
 -> Waiting for data to render...
 [OK] Data retrieved successfully.


{'datetime': '2026-05-02 12:49:08',
 'post': {'new_used': 'Used',
  'title': '2020 Subaru Forester Touring',
  'mileage': 88837,
  'price': 19998,
  'monthly_payment': 361,
  'basic_desc': {'VIN': 'JF2SKAXC6LH413248',
   'Stock Number': 'SU24874XA',
   'Exterior Color': 'Horizon Blue Pearl',
   'Interior Color': 'Black',
   'Fuel Type': 'Gasoline',
   'Engine': '2.5L H4 16V GDI DOHC',
   'MPG': '26-33',
   'Drivetrain': 'All-wheel Drive',
   'Transmission': 'Automatic CVT'},
  'feature_des': {'Convenience': ['Adaptive Cruise Control',
    'Heated Seats',
    'Heated Steering Wheel',
    'Keyless Entry',
    'Navigation System',
    'Power Liftgate'],
   'Entertainment': ['Apple CarPlay®',
    'Bluetooth®',
    'CD Player',
    'HomeLink',
    'Premium Sound System',
    'Satellite Radio'],
   'Exterior': ['Alloy Wheels', 'Roof Rack', 'Sunroof/Moonroof'],
   'Safety': ['Backup Camera', 'Brake Assist', 'Stability Control'],
   'Seating': ['Leather Seats', 'Memory Seat']},
  'user_history

In [53]:
B = "https://www.cars.com/vehicledetail/c77f5f02-3593-4151-9605-10b076d6b228/"

scrape_full_info(B)

Attempt 1/5: Navigating to https://www.cars.com/vehicledetail/c77f5f02-3593-4151-9605-10b076d6b228/...
 [!] Blocked by Cloudflare on attempt 1. Exiting to retry...
 [X] Error: Cloudflare Blocked
 -> Resting for 2.07s before re-initializing driver...

Attempt 2/5: Navigating to https://www.cars.com/vehicledetail/c77f5f02-3593-4151-9605-10b076d6b228/...
 [!] Blocked by Cloudflare on attempt 2. Exiting to retry...
 [X] Error: Cloudflare Blocked
 -> Resting for 2.35s before re-initializing driver...

Attempt 3/5: Navigating to https://www.cars.com/vehicledetail/c77f5f02-3593-4151-9605-10b076d6b228/...
 [!] Blocked by Cloudflare on attempt 3. Exiting to retry...
 [X] Error: Cloudflare Blocked
 -> Resting for 2.37s before re-initializing driver...

Attempt 4/5: Navigating to https://www.cars.com/vehicledetail/c77f5f02-3593-4151-9605-10b076d6b228/...
 [!] Blocked by Cloudflare on attempt 4. Exiting to retry...
 [X] Error: Cloudflare Blocked
 -> Resting for 2.78s before re-initializing driver.

{'datetime': '2026-05-02 12:51:11',
 'post': {'new_used': 'Certified',
  'title': '2024 Acura MDX Technology',
  'mileage': 26689,
  'price': 38888,
  'monthly_payment': 703,
  'basic_desc': {'VIN': '5J8YE1H47RL021085',
   'Stock Number': 'M9155A',
   'Exterior Color': 'Majestic Black Pearl',
   'Interior Color': 'Ebony',
   'Fuel Type': 'Gasoline',
   'Engine': '3.5L V6 24V GDI SOHC',
   'MPG': '19-25',
   'Drivetrain': 'All-wheel Drive',
   'Transmission': '10-Speed Automatic'},
  'feature_des': {'Convenience': ['Adaptive Cruise Control',
    'Heated Seats',
    'Keyless Entry',
    'Keyless Start',
    'Navigation System',
    'Power Folding Mirrors',
    'Power Liftgate'],
   'Entertainment': ['Bluetooth®',
    'HomeLink',
    'Premium Sound System',
    'Satellite Radio',
    'WiFi Hotspot'],
   'Exterior': ['Alloy Wheels', 'Moonroof'],
   'Safety': ['Backup Camera',
    'Blind Spot Monitor',
    'Brake Assist',
    'LED Headlights',
    'Lane Departure Warning',
    'Rain Sensing

In [55]:
import os
import re
import time
import random
from typing import Optional, List, Set
from urllib.parse import urljoin

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


RESULTS_URL_TMPL = (
    "https://www.cars.com/shopping/results/?deal_ratings%5B%5D=good&zip=60606&maximum_distance=9999&sort=best_match_desc&page={page}"
)
SITE_BASE = "https://www.cars.com"


def init_driver(headless: bool = True, driver_path: Optional[str] = None) -> webdriver.Chrome:
    """Initialize Selenium Chrome WebDriver with sensible defaults."""
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")  # modern headless
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--blink-settings=imagesEnabled=false")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
    )

    if driver_path:
        service = Service(driver_path)
        driver = webdriver.Chrome(service=service, options=chrome_options)
    else:
        driver = webdriver.Chrome(options=chrome_options)
    return driver


def wait_for_cards(driver: webdriver.Chrome, timeout: int = 15) -> None:
    """Wait until at least one vehicle card link is present in the DOM."""
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.data-card-link"))
    )


def scroll_to_bottom(driver: webdriver.Chrome, pause: float = 0.6, max_rounds: int = 12) -> None:
    """Scroll down to trigger lazy loading."""
    last_height = driver.execute_script("return document.body.scrollHeight")
    rounds = 0
    while rounds < max_rounds:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
        rounds += 1


def extract_links_from_html(html: str) -> List[str]:
    """Parse Cars.com results page HTML and extract all vehicle detail links."""
    soup = BeautifulSoup(html, "html.parser")
    tags = soup.select("a[data-card-link]")
    links: List[str] = []
    for tag in tags:
        href = tag.get("href")
        if href and "/vehicledetail/" in href:
            links.append(urljoin(SITE_BASE, href))
    return links


def crawl_all_listing_urls(
    start_page: int = 1,
    end_page: int = 3,
    output_dir: str = "car_links",
    delay_between_pages: float = 2.0,
    headless: bool = True,
    driver_path: Optional[str] = None,
    clear_output_first: bool = False,
) -> None:
    """
    Crawl Cars.com listings across multiple pages and store URLs page-by-page.

    Each page will have its own file, e.g. car_links/page_1.txt, car_links/page_2.txt, etc.
    """
    os.makedirs(output_dir, exist_ok=True)

    if clear_output_first:
        for file in os.listdir(output_dir):
            if file.startswith("page_") and file.endswith(".txt"):
                os.remove(os.path.join(output_dir, file))

    driver = init_driver(headless=headless, driver_path=driver_path)
    try:
        for page in range(start_page, end_page + 1):
            url = RESULTS_URL_TMPL.format(page=page)
            print(f"Fetching page {page}: {url}")

            try:
                driver.get(url)
            except Exception as e:
                print(f"Error navigating to page {page}: {e}")
                continue

            try:
                wait_for_cards(driver, timeout=20)
            except Exception:
                pass

            scroll_to_bottom(driver, pause=0.6, max_rounds=12)
            links = extract_links_from_html(driver.page_source)

            if not links:
                print(f"No car links found on page {page}")
                continue

            output_file = os.path.join(output_dir, f"page_{page}.txt")

            with open(output_file, "w", encoding="utf-8") as f_out:
                for lnk in links:
                    f_out.write(lnk + "\n")

            print(f"Saved {len(links)} links to {output_file}")

            time.sleep(delay_between_pages + random.uniform(0.3, 1.2))

    finally:
        driver.quit()
        print(f"Done. All links saved to folder '{output_dir}'.")


if __name__ == "__main__":
    start_page = 1
    end_page = 1
    crawl_all_listing_urls(
        start_page=start_page,
        end_page=end_page,
        output_dir="car_links",
        delay_between_pages=2.0,
        headless=True,
        driver_path=None,  # or specify "/usr/lib/chromium-browser/chromedriver"
        clear_output_first=False,
    )


Fetching page 1: https://www.cars.com/shopping/results/?deal_ratings%5B%5D=good&zip=60606&maximum_distance=9999&sort=best_match_desc&page=1
Saved 28 links to car_links/page_1.txt
Done. All links saved to folder 'car_links'.


In [57]:
import os
import json
import time
import random
from typing import List

def read_urls_from_file(file_path: str) -> List[str]:
    """Read all non-empty lines (URLs) from a file."""
    if not os.path.exists(file_path):
        print(f"URL file not found: {file_path}")
        return []
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def scrape_from_url_files(
    link_folder: str = "car_links",
    output_root: str = "raw_data",
    delay: float = 2.0,
    headless: bool = True,
    from_page: int = 1,
    to_page: int = 3,
):
    """
    Reads car link files (per page) and scrapes data for each URL,
    storing one JSON per car under raw_data/<page_num>/<index>.json

    Args:
        link_folder: Directory containing page_X.txt files.
        output_root: Root directory to store raw JSON data.
        delay: Delay between scraping each URL.
        headless: Run Chrome in headless mode.
        from_page: Starting page number to scrape.
        to_page: Ending page number to scrape (inclusive).
    """
    os.makedirs(output_root, exist_ok=True)

    # Construct page range
    for page_number in range(from_page, to_page + 1):
        file_name = f"page_{page_number}.txt"
        file_path = os.path.join(link_folder, file_name)

        if not os.path.exists(file_path):
            print(f"File not found: {file_name} — skipping.")
            continue

        page_output_dir = os.path.join(output_root, str(page_number))
        os.makedirs(page_output_dir, exist_ok=True)

        urls = read_urls_from_file(file_path)
        print(f"Found {len(urls)} URLs in {file_name}")

        for i, url in enumerate(urls, start=1):
            output_path = os.path.join(page_output_dir, f"{i}.json")

            if os.path.exists(output_path):
                print(f"Skipping existing file: {output_path}")
                continue

            print(f"[Page {page_number}] Scraping car {i}/{len(urls)}: {url}")
            try:
                data = scrape_full_info(url, headless=headless)
                if not data:
                    print(f"Empty data for {url}, skipping.")
                    continue

                with open(output_path, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)

                print(f"Saved -> {output_path}")

            except Exception as e:
                print(f"Error scraping {url}: {e}")

            time.sleep(delay + random.uniform(0.5, 1.5))

    print(f"All scraping done for pages {from_page} to {to_page}.")


if __name__ == "__main__":
    # Example: scrape pages 2 through 5 only
    scrape_from_url_files(
        link_folder="car_links",    # Folder containing page_X.txt files
        output_root="raw_data",     # Folder where JSON data will go
        delay=2.5,
        headless=True,
        from_page=1,                # Start page
        to_page=2                   # End page
    )


Found 28 URLs in page_1.txt
[Page 1] Scraping car 1/28: https://www.cars.com/vehicledetail/02eb22ed-d5a2-4b4e-a244-f441dcc83f92/?attribution_type=p_one
Attempt 1/20: Navigating to https://www.cars.com/vehicledetail/02eb22ed-d5a2-4b4e-a244-f441dcc83f92/?attribution_type=p_one...
 -> Waiting for data to render...
 [OK] Data retrieved successfully.
Attempt 1/20: Navigating to https://www.cars.com/dealers/2622/sheehy-ford-of-springfield/#Reviews...
 -> Waiting for data to render...
 [OK] Data retrieved successfully.
Saved -> raw_data/1/1.json
[Page 1] Scraping car 2/28: https://www.cars.com/vehicledetail/75d34cfc-eef0-4f1d-9b88-ad2a2a18e6d8/?attribution_type=isa
Attempt 1/20: Navigating to https://www.cars.com/vehicledetail/75d34cfc-eef0-4f1d-9b88-ad2a2a18e6d8/?attribution_type=isa...
 [!] Blocked by Cloudflare on attempt 1. Exiting to retry...
 [X] Error: Cloudflare Blocked
 -> Resting for 3.62s before re-initializing driver...

Attempt 2/20: Navigating to https://www.cars.com/vehicledeta

KeyboardInterrupt: 